# Session VII: Command Line Tools and Automation

Use Python to automate tasks and interact through the command line, including parsing arguments and running scripts from the terminal.

## Exercise 1: CLI Greeter and Calculator
**Objective:** Practice basic command-line parsing using a list that represents `sys.argv`.

Instead of reading real terminal input in the notebook, you will write a function that receives an argument list like:

```python
["greet", "Alice"]
["add", "3", "4"]
```

**Tasks:**
1. `run_command(argv)` should support two commands:
   - `greet <name>` → return `"Hello, <name>!"`
   - `add <x> <y>` → return the numeric sum as a string
2. Return a helpful usage message for missing or invalid input.
3. Use `try/except` to handle non-numeric values in `add`.


In [1]:
def run_command(argv):
    """Parse a simple command list and return a response string."""
    usage = (
        "Usage:\n"
        "  greet <name>    - Greet someone by name\n"
        "  add <x> <y>     - Add two numbers together"
    )

    # TODO: Handle empty input
    if not argv:
        return f"Error: No command provided.\n{usage}"

    command = argv[0].lower()

    # TODO: Support: greet <name>
    if command == "greet":
        if len(argv) != 2:
            return f"Error: 'greet' expects exactly 1 argument (<name>).\nUsage: greet <name>"
        name = argv[1]
        return f"Hello, {name}!"

    # TODO: Support: add <x> <y>
    elif command == "add":
        if len(argv) != 3:
            return f"Error: 'add' expects exactly 2 arguments (<x> <y>).\nUsage: add <x> <y>"
    # TODO: Use try/except for numeric conversion
        try:
            x = float(argv[1])
            y = float(argv[2])
            return str(x + y)
        except ValueError:
            return f"Error: Both arguments for 'add' must be valid numbers. Received: {argv[1]!r}, {argv[2]!r}"

    # TODO: Return an error/usage message for bad commands
    else:
        return f"Error: Unknown command '{argv[0]}'.\n{usage}"


# --- TEST ---
print(run_command(["greet", "Alice"]))
# Output: Hello, Alice!

print(run_command(["add", "3", "4"]))
# Output: 7.0

print(run_command(["add", "three", "4"]))
# Output: Error: Both arguments for 'add' must be valid numbers. Received: 'three', '4'

print(run_command([]))
# Output: Error: No command provided. ...

print(run_command(["multiply", "2", "3"]))
# Output: Error: Unknown command 'multiply'. ...

Hello, Alice!
7.0
Error: Both arguments for 'add' must be valid numbers. Received: 'three', '4'
Error: No command provided.
Usage:
  greet <name>    - Greet someone by name
  add <x> <y>     - Add two numbers together
Error: Unknown command 'multiply'.
Usage:
  greet <name>    - Greet someone by name
  add <x> <y>     - Add two numbers together


---
## Exercise 2: Expense Lister with `argparse`
**Objective:** Build a real argument parser using optional flags.

You are given a list of expense records. Build a command-line style filter that can:
- show all expenses
- filter by category
- filter by a minimum amount

**Tasks:**
1. `build_parser()` → Create an `ArgumentParser` with:
   - `--category`
   - `--min` (float, default `0`)
2. `filter_expenses(expenses, category=None, min_amount=0)` → Return only matching expenses.
3. `format_expenses(expenses)` → Return a formatted multi-line string.


In [3]:
import argparse

expenses = [
    {"date": "2026-10-01", "category": "Food", "amount": 18.50, "note": "Lunch"},
    {"date": "2026-10-02", "category": "Transport", "amount": 2.80, "note": "Metro"},
    {"date": "2026-10-02", "category": "Food", "amount": 34.20, "note": "Groceries"},
    {"date": "2026-10-03", "category": "Books", "amount": 45.00, "note": "Python book"},
]


def build_parser():
    """Return an argparse parser for expense filtering."""
    parser = argparse.ArgumentParser(description="Filter and list expense records.")
    parser.add_argument(
        "--category",
        type=str,
        default=None,
        help="Filter expenses by category (case-insensitive)",
    )
    parser.add_argument(
        "--min",
        dest="min",
        type=float,
        default=0.0,
        help="Minimum expense amount (default: 0.0)",
    )
    return parser


def filter_expenses(expenses, category=None, min_amount=0):
    """Return filtered expenses matching the given criteria."""
    result = []
    for exp in expenses:
        # Check minimum amount threshold
        if exp["amount"] < min_amount:
            continue

        # Check category match (case-insensitive comparison)
        if category is not None and exp["category"].lower() != category.lower():
            continue

        result.append(exp)
    return result


def format_expenses(expenses):
    """Return a formatted text report for a list of expenses."""
    if not expenses:
        return "No expenses found matching the criteria."

    # Header and divider
    header = f"{'Date':<12} {'Category':<12} {'Amount':>8}   {'Note'}"
    divider = "-" * len(header)
    lines = [header, divider]

    total = 0.0
    for exp in expenses:
        total += exp["amount"]
        lines.append(
            f"{exp['date']:<12} {exp['category']:<12} {exp['amount']:>8.2f}   {exp['note']}"
        )

    lines.append(divider)
    lines.append(f"{'Total':<25} {total:>8.2f}")

    return "\n".join(lines)


# --- TEST ---
parser = build_parser()
args = parser.parse_args(["--category", "Food", "--min", "20"])
filtered = filter_expenses(expenses, args.category, args.min)
print(format_expenses(filtered))


Date         Category       Amount   Note
-----------------------------------------
2026-10-02   Food            34.20   Groceries
-----------------------------------------
Total                        34.20


---
## Exercise 3: Budget Backup Script
**Objective:** Automate data backups using `pathlib`, timestamps, and file copying.

Your script should validate that the source file exists, create a `backups` folder if needed, and make a timestamped copy.

**Tasks:**
1. `make_backup_name(filename)` → Return a new filename like `budget_20261005_224700.json`
2. `backup_file(source_path, backup_dir="backups")` → Copy the source file into the backup folder.
3. Return the destination path as a string, or `None` if the source file is missing.

**First, run the setup cell below to create a sample file.**


In [4]:
from pathlib import Path
import json

sample_budget = [
    {"category": "Food", "amount": 20.5},
    {"category": "Transport", "amount": 15.0},
]

with open("budget_data.json", "w") as file:
    json.dump(sample_budget, file, indent=2)

print("Created budget_data.json")


Created budget_data.json


In [5]:
from pathlib import Path
from datetime import datetime
import shutil


def make_backup_name(filename):
    """Return a timestamped backup filename."""
    path = Path(filename)
    stem = path.stem
    suffix = path.suffix

    # TODO: Add a timestamp with datetime.now().strftime(...)
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")

    return f"{stem}_{timestamp}{suffix}"


def backup_file(source_path, backup_dir="backups"):
    """Copy a file into backup_dir with a timestamped name."""
    src = Path(source_path)
    dest_dir = Path(backup_dir)

    # TODO: Check whether source exists
    if not src.is_file():
        raise FileNotFoundError(f"Source file not found: {src.resolve()}")

    # TODO: Create backup folder if needed.mkdir(parents=True, exist_ok=True)

    # Generate backup filename and full destination path
    dest_filename = make_backup_name(src.name)
    dest_path = dest_dir / dest_filename

    # TODO: Copy file with shutil.copy2
    shutil.copy2(src, dest_path)

    # TODO: Return destination path as a string
    return str(dest_path)


# --- TEST ---
result = backup_file("budget_data.json")
print("Backup created:", result)


Backup created: backups/budget_data_20261111_081000.json


---
## Exercise 4: CSV Expense Summary CLI
**Objective:** Load a CSV file, validate rows, and print category totals.

**Tasks:**
1. `load_expenses_csv(filename)` → Use `csv.DictReader` to load rows.
2. Skip malformed rows where:
   - `category` is empty
   - `amount` is missing or non-numeric
3. `summarize_by_category(expenses)` → Return a dictionary of totals.
4. `print_summary(summary)` → Print a formatted report.

**Run the setup cell below first.**


In [6]:
import csv

rows = [
    ["date", "category", "amount", "note"],
    ["2026-10-01", "Food", "18.50", "Lunch"],
    ["2026-10-02", "Transport", "2.80", "Metro"],
    ["2026-10-02", "", "12.00", "Broken row"],
    ["2026-10-03", "Books", "forty", "Invalid amount"],
    ["2026-10-04", "Food", "21.30", "Dinner"],
]

with open("expenses_cli.csv", "w", newline="") as file:
    writer = csv.writer(file)
    writer.writerows(rows)

print("Created expenses_cli.csv")


Created expenses_cli.csv


In [7]:
import csv


def load_expenses_csv(filename):
    """Load valid expense rows from a CSV file."""
    valid_rows = []

    with open(filename, mode="r", newline="", encoding="utf-8") as file:
        reader = csv.DictReader(file)
        for row in reader:
            category = (row.get("category") or "").strip()
            amount_str = (row.get("amount") or "").strip()

            if not category:
                continue


            try:
                amount = float(amount_str)
            except ValueError:
                continue

            row["category"] = category
            row["amount"] = amount
            valid_rows.append(row)

    return valid_rows


def summarize_by_category(expenses):
    """Return {category: total_amount}."""
    totals = {}
    for exp in expenses:
        cat = exp["category"]
        totals[cat] = totals.get(cat, 0.0) + exp["amount"]
    return totals


def print_summary(summary):
    """Print a formatted category summary."""
    if not summary:
        print("No expense data available to summarize.")
        return

    header = f"{'Category':<15} {'Total Amount':>12}"
    divider = "-" * len(header)
    print(header)
    print(divider)

    grand_total = 0.0
    for category, total in sorted(summary.items()):
        grand_total += total
        print(f"{category:<15} {total:>12.2f}")

    print(divider)
    print(f"{'Grand Total':<15} {grand_total:>12.2f}")


# --- TEST ---
expenses = load_expenses_csv("expenses_cli.csv")
summary = summarize_by_category(expenses)
print_summary(summary)


Category        Total Amount
----------------------------
Food                   39.80
Transport               2.80
----------------------------
Grand Total            42.60


---
## Exercise 5: Scheduling Helper
**Objective:** Generate schedule strings for cron and Windows Task Scheduler.

You are **not** actually scheduling tasks in this notebook. Instead, you will generate the correct command strings.

**Tasks:**
1. `build_cron_schedule(minute, hour, script_path)` → Return a cron line like:
   `30 18 * * * python /path/to/script.py`
2. `build_task_scheduler_command(time_str, script_path)` → Return a Windows-style command string.
3. Validate the time values and raise or handle `ValueError` for invalid input.


In [8]:
from datetime import datetime
from pathlib import Path


def build_cron_schedule(minute, hour, script_path):
    """Return a cron schedule line for a daily script."""
    # Validate minute (0-59) and hour (0-23)
    try:
        minute_int = int(minute)
        hour_int = int(hour)
    except (ValueError, TypeError):
        raise ValueError("Minute and hour must be integers.")

    if not (0 <= minute_int <= 59):
        raise ValueError(f"Invalid minute: {minute}. Must be between 0 and 59.")
    if not (0 <= hour_int <= 23):
        raise ValueError(f"Invalid hour: {hour}. Must be between 0 and 23.")

    clean_path = str(script_path).strip()
    if not clean_path:
        raise ValueError("script_path cannot be empty.")

    # Format 
    return f"{minute_int} {hour_int} * * * python {clean_path}"


def build_task_scheduler_command(time_str, script_path, task_name=None):
    """Return a Windows schtasks command string."""
    clean_time = str(time_str).strip()

    # Validate HH:MM 24-hour format
    try:
        parsed_time = datetime.strptime(clean_time, "%H:%M")
        formatted_time = parsed_time.strftime("%H:%M")
    except ValueError:
        raise ValueError(
            f"Invalid time format: '{time_str}'. Expected 24-hour 'HH:MM' (00:00 to 23:59)."
        )

    clean_path = str(script_path).strip()
    if not clean_path:
        raise ValueError("script_path cannot be empty.")

    # Use filename stem as task name fallback if none provided
    if not task_name:
        task_name = Path(clean_path).stem or "ScheduledScript"

    # Convert forward slashes to Windows-standard backslashes for script_path
    windows_path = clean_path.replace("/", "\\")

    # Command: schtasks /Create /SC DAILY /TN <TaskName> /TR "python <path>" /ST <HH:MM>
    return (
        f'schtasks /Create /SC DAILY /TN "{task_name}" '
        f'/TR "python \\"{windows_path}\\"" /ST {formatted_time}'
    )


# --- TEST ---
print(build_cron_schedule(30, 18, "/home/user/backup.py"))
print(build_task_scheduler_command("18:30", "C:/scripts/backup.py"))


30 18 * * * python /home/user/backup.py
schtasks /Create /SC DAILY /TN "backup" /TR "python \"C:\scripts\backup.py\"" /ST 18:30


---
## Exercise 6: Automation Suite
**Objective:** Combine parsing, loading JSON, listing expenses, backups, and summaries into one mini CLI application.

Build a tool with these subcommands:
- `list` → print all expenses
- `summary` → show totals by category
- `backup` → make a timestamped backup of the JSON file

**Tasks:**
1. `load_json_expenses(filename)` → Safely load a JSON list of expense dictionaries.
2. `build_suite_parser()` → Create an `argparse` parser with subcommands:
   - `list`
   - `summary`
   - `backup`
   Also add a common `filename` argument.
3. `run_suite(argv=None)` → Parse the arguments and perform the correct action.
4. Reuse functions from earlier exercises when possible.

**Run the setup cell below first.**


In [9]:
import json

suite_data = [
    {"date": "2026-10-01", "category": "Food", "amount": 18.50, "note": "Lunch"},
    {"date": "2026-10-02", "category": "Transport", "amount": 2.80, "note": "Metro"},
    {"date": "2026-10-03", "category": "Food", "amount": 21.30, "note": "Dinner"},
    {"date": "2026-10-04", "category": "Books", "amount": 45.00, "note": "Python book"},
]

with open("expenses_suite.json", "w") as file:
    json.dump(suite_data, file, indent=2)

print("Created expenses_suite.json")


Created expenses_suite.json


In [10]:
import argparse
from datetime import datetime
import json
from pathlib import Path
import shutil

def load_json_expenses(filename):
    """Safely load and validate a JSON expense list."""
    try:
        with open(filename, "r", encoding="utf-8") as file:
            data = json.load(file)
            if not isinstance(data, list):
                print(f"Warning: Expected a JSON array in '{filename}'.")
                return []
            return data
    except (FileNotFoundError, json.JSONDecodeError, OSError) as e:
        print(f"Error loading file '{filename}': {e}")
        return []


def build_suite_parser():
    """Return an argparse parser with subcommands."""
    parser = argparse.ArgumentParser(
        description="Expense automation CLI suite",
        prog="expense_suite",
    )

    # Positional filename argument before subcommands
    parser.add_argument("filename", help="Path to the JSON expense file")

    # Subcommands: list, summary, backup
    subparsers = parser.add_subparsers(
        dest="command",
        required=True,
        help="Subcommand to execute",
    )

    subparsers.add_parser("list", help="List all expenses in tabular format")
    subparsers.add_parser("summary", help="Show total expenses grouped by category")
    subparsers.add_parser("backup", help="Create a timestamped backup of the file")

    return parser


def run_suite(argv=None):
    """Run the automation suite and return a message or report."""
    parser = build_suite_parser()


    try:
        args = parser.parse_args(argv)
    except SystemExit:
        return "Command parsing failed or help displayed."


    if args.command == "backup":
        src = Path(args.filename)
        if not src.is_file():
            return f"Backup failed: Source file not found: {src.resolve()}"

        dest_dir = Path("backups")
        dest_dir.mkdir(parents=True, exist_ok=True)

        timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
        backup_filename = f"{src.stem}_{timestamp}{src.suffix}"
        dest_path = dest_dir / backup_filename

        shutil.copy2(src, dest_path)
        return f"Backup created successfully: {dest_path}"

    # Load data for 'list' and 'summary'
    expenses = load_json_expenses(args.filename)
    if not expenses:
        return f"No records found or file '{args.filename}' could not be loaded."


    if args.command == "list":
        header = f"{'Date':<12} {'Category':<12} {'Amount':>8}   {'Note'}"
        divider = "-" * len(header)
        lines = [header, divider]

        total = 0.0
        for exp in expenses:
            try:
                amt = float(exp.get("amount", 0.0))
            except (ValueError, TypeError):
                amt = 0.0

            total += amt
            lines.append(
                f"{exp.get('date', 'N/A'):<12} "
                f"{exp.get('category', 'N/A'):<12} "
                f"{amt:>8.2f}   "
                f"{exp.get('note', '')}"
            )

        lines.append(divider)
        lines.append(f"{'Total':<25} {total:>8.2f}")
        return "\n".join(lines)

    elif args.command == "summary":
        totals = {}
        for exp in expenses:
            cat = exp.get("category") or "Uncategorized"
            try:
                amt = float(exp.get("amount", 0.0))
            except (ValueError, TypeError):
                continue
            totals[cat] = totals.get(cat, 0.0) + amt

        header = f"{'Category':<15} {'Total Amount':>12}"
        divider = "-" * len(header)
        lines = [header, divider]

        grand_total = 0.0
        for category, cat_total in sorted(totals.items()):
            grand_total += cat_total
            lines.append(f"{category:<15} {cat_total:>12.2f}")

        lines.append(divider)
        lines.append(f"{'Grand Total':<15} {grand_total:>12.2f}")
        return "\n".join(lines)

    return f"Unknown command '{args.command}'."


# --- TEST ---
print(run_suite(["expenses_suite.json", "list"]))
print()
print(run_suite(["expenses_suite.json", "summary"]))
print()
print(run_suite(["expenses_suite.json", "backup"]))


Date         Category       Amount   Note
-----------------------------------------
2026-10-01   Food            18.50   Lunch
2026-10-02   Transport        2.80   Metro
2026-10-03   Food            21.30   Dinner
2026-10-04   Books           45.00   Python book
-----------------------------------------
Total                        87.60

Category        Total Amount
----------------------------
Books                  45.00
Food                   39.80
Transport               2.80
----------------------------
Grand Total            87.60

Backup created successfully: backups/expenses_suite_20261111_081928.json
